# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshiniChebrolu/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### My baseline rule

I use a simple, transparent action rule based on observed page staleness and search visibility.

- **REFRESH**: prioritize pages that are sufficiently stale and show weaker visibility.
- **CTR_FIX**: prioritize pages where search visibility is present but the observed CTR is relatively weak for the available position information.
- **MONITOR**: use when neither stronger action condition is met.

The score is designed only for decision-support. It uses information available for the current observation and does not use future-window trend fields or product flags.

### Reason codes

- `STALE_REFRESH` — the page shows a stronger staleness signal and is selected for refresh.
- `CTR_POSITION` — the page shows a CTR-versus-position signal and is selected for CTR investigation.
- `MONITOR_BASELINE` — neither action signal is sufficiently strong for the baseline rule.

In [5]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Saving content_refresh_anonymized (2).csv to content_refresh_anonymized (2).csv
Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the fields used by the baseline rule.

required_cols = ["days_since_update", "avg_position", "ctr"]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)

print("\nRows:", len(df))
print("Rule inputs:", required_cols)

Missing required columns: ['days_since_update']

Rows: 30000
Rule inputs: ['days_since_update', 'avg_position', 'ctr']


## 2. Build the ranked queue (writes the CSV)

### Baseline scoring rule

The baseline gives each row a transparent action score using two observable signals:

1. **Staleness** — higher days since update increases the refresh score.
2. **CTR versus position** — weaker CTR relative to the page's observed average position increases the CTR-fix score.

The action is assigned from the stronger signal. The resulting queue is ranked by the action score.

This is a simple baseline rather than a predictive ML model. Its purpose is to provide an interpretable decision-support queue that can later be compared with a Week-5 model.

In [8]:
import numpy as np
import pandas as pd

work = df.copy()

# Convert rule inputs to numeric
work["days_since_last_update"] = pd.to_numeric(
    work["days_since_last_update"], errors="coerce"
)
work["avg_position"] = pd.to_numeric(
    work["avg_position"], errors="coerce"
)
work["ctr"] = pd.to_numeric(
    work["ctr"], errors="coerce"
)

# Staleness signal: higher = more stale
work["staleness_signal"] = work["days_since_last_update"].rank(
    pct=True, method="average"
)

# CTR signal: lower CTR = stronger CTR-fix signal
work["ctr_signal"] = 1 - work["ctr"].rank(
    pct=True, method="average"
)

# Combined transparent baseline score
work["action_score"] = (
    0.55 * work["staleness_signal"]
    + 0.45 * work["ctr_signal"]
)

# Assign action
work["action"] = np.select(
    [
        work["staleness_signal"] >= 0.75,
        work["ctr_signal"] >= 0.75
    ],
    [
        "REFRESH",
        "CTR_FIX"
    ],
    default="MONITOR"
)

# Assign reason code
work["reason_code"] = np.select(
    [
        work["action"].eq("REFRESH"),
        work["action"].eq("CTR_FIX")
    ],
    [
        "STALE_REFRESH",
        "CTR_POSITION"
    ],
    default="MONITOR_BASELINE"
)

# Rank the queue
work = work.sort_values(
    ["action_score", "days_since_last_update"],
    ascending=[False, False]
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)

# Required output
baseline_queue = work[
    [
        "rank",
        "content_id",
        "action_score",
        "action",
        "reason_code",
        "days_since_last_update",
        "avg_position",
        "ctr"
    ]
].copy()

baseline_queue.to_csv(
    "baseline_action_score.csv",
    index=False
)

print("Rows written:", len(baseline_queue))
print("\nAction counts:")
print(baseline_queue["action"].value_counts())

baseline_queue.head(10)

Rows written: 30000

Action counts:
action
MONITOR    10693
CTR_FIX    10216
REFRESH     9091
Name: count, dtype: int64


,rank,content_id,action_score,action,reason_code,days_since_last_update,avg_position,ctr
0,1,content_55a5b1c46474,0.900884,REFRESH,STALE_REFRESH,373,7.5,0.0
1,2,content_f6fdf87348f6,0.900884,REFRESH,STALE_REFRESH,373,32.5,0.0
2,3,content_8d56efff1e71,0.900838,REFRESH,STALE_REFRESH,372,35.0,0.0
3,4,content_1b4ec72dafd4,0.900838,REFRESH,STALE_REFRESH,372,7.0,0.0
4,5,content_e2b702f4f92b,0.900783,REFRESH,STALE_REFRESH,334,9.3,0.0
5,6,content_06e19c6486b0,0.900783,REFRESH,STALE_REFRESH,334,5.0,0.0
6,7,content_7a888d3d99c8,0.900728,REFRESH,STALE_REFRESH,313,67.6,0.0
7,8,content_6476d1d8c050,0.900728,REFRESH,STALE_REFRESH,313,67.8,0.0
8,9,content_94991fe6268c,0.900728,REFRESH,STALE_REFRESH,313,12.4,0.0
9,10,content_02b0d6e30129,0.900728,REFRESH,STALE_REFRESH,313,6.9,0.0


## 3. Top-20 review

### Top-20 review

The top 20 rows are reviewed as decision-support recommendations rather than certain decisions.

For each row, I record the action, reason code, confidence note, and what additional evidence could make the recommendation wrong.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 review

top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    if row["action"] == "REFRESH":
        return "Higher confidence because the staleness signal is strong."
    elif row["action"] == "CTR_FIX":
        return "Moderate confidence because the CTR signal is strong; position and query context should be checked."
    else:
        return "Lower confidence because no single action signal is especially strong."

def what_would_make_it_wrong(row):
    if row["action"] == "REFRESH":
        return "Could be wrong if the page was intentionally left unchanged or the update date is not representative."
    elif row["action"] == "CTR_FIX":
        return "Could be wrong if CTR is explained by query mix, SERP features, or position context."
    else:
        return "Could be wrong if an important signal is missing from this simple baseline."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong, axis=1
)

review_cols = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "action_score",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_cols]

display(top20_review)

,rank,content_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,1,content_55a5b1c46474,REFRESH,STALE_REFRESH,0.900884,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
1,2,content_f6fdf87348f6,REFRESH,STALE_REFRESH,0.900884,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
2,3,content_8d56efff1e71,REFRESH,STALE_REFRESH,0.900838,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
3,4,content_1b4ec72dafd4,REFRESH,STALE_REFRESH,0.900838,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
4,5,content_e2b702f4f92b,REFRESH,STALE_REFRESH,0.900783,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
5,6,content_06e19c6486b0,REFRESH,STALE_REFRESH,0.900783,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
6,7,content_7a888d3d99c8,REFRESH,STALE_REFRESH,0.900728,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
7,8,content_6476d1d8c050,REFRESH,STALE_REFRESH,0.900728,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
8,9,content_94991fe6268c,REFRESH,STALE_REFRESH,0.900728,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...
9,10,content_02b0d6e30129,REFRESH,STALE_REFRESH,0.900728,Higher confidence because the staleness signal...,Could be wrong if the page was intentionally l...


## 4. Weak picks + leakage check

### Weak picks

The baseline is intentionally simple, so some high-ranked rows can be false positives.

A refresh recommendation could be wrong when a page is intentionally not updated or when the recorded update date does not represent meaningful content changes.

A CTR-fix recommendation could be wrong because CTR can depend on query mix, search-result features, intent, and position context that are not fully captured by this baseline.

### Leakage check

The baseline score and action assignment use only current observable signals: staleness, average position, and CTR.

Future/outcome fields such as `trend_direction` and `trend_pct` are not used to construct the score or action.

Product flags are also not used as scoring inputs.

The resulting queue is therefore intended as a simple decision-support baseline rather than a future-aware model.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check

scoring_fields = [
    "days_since_last_update",
    "avg_position",
    "ctr"
]

future_fields = [
    "trend_direction",
    "trend_pct"
]

print("Fields used for scoring:")
for col in scoring_fields:
    print(" -", col)

print("\nFuture/outcome fields:")
for col in future_fields:
    print(
        f" - {col}:",
        "NOT USED IN SCORING" if col not in scoring_fields
        else "WARNING"
    )

# Check possible product/flag columns
possible_flag_columns = [
    col for col in df.columns
    if "flag" in col.lower()
]

print("\nPossible flag-related columns:")
print(possible_flag_columns)

print("\nLeakage conclusion:")
print(
    "No future/outcome fields or product flag fields are used "
    "to calculate action_score or action."
)

Fields used for scoring:
 - days_since_last_update
 - avg_position
 - ctr

Future/outcome fields:
 - trend_direction: NOT USED IN SCORING
 - trend_pct: NOT USED IN SCORING

Possible flag-related columns:
[]

Leakage conclusion:
No future/outcome fields or product flag fields are used to calculate action_score or action.


In [11]:
# Quick sanity check

print("Top-20 rows:", len(top20_review))
print("Total ranked rows:", len(baseline_queue))
print("\nAction distribution:")
print(baseline_queue["action"].value_counts())

print("\nBaseline queue successfully created.")

Top-20 rows: 20
Total ranked rows: 30000

Action distribution:
action
MONITOR    10693
CTR_FIX    10216
REFRESH     9091
Name: count, dtype: int64

Baseline queue successfully created.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.